# SecureOps — Python Security & Data Utility Toolkit

A desktop application built with Python and Tkinter. It includes file analysis, CSV management, an encrypted local vault, and stack/queue visualization.

## How to use this notebook
1. Run the installation cell.
2. Run the source-setup cell to write the complete application to `secureops.py` in the current folder.
3. Run the launch cell. The Tkinter window opens separately on a local desktop that supports GUI applications.

**Note:** This notebook contains the complete application source in the source-setup cell. Tkinter needs a local desktop GUI; some hosted notebook environments do not support GUI windows.

In [ ]:
%pip install cryptography

## Complete SecureOps application source

Run this cell to create `secureops.py` from the full source code included below.

In [ ]:
from pathlib import Path
source_code = r'''import os
import re
import csv
import json
import base64
import secrets
import threading
import queue
from pathlib import Path
from collections import Counter
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC


APP_NAME = "SecureOps"
VAULT_FILE = Path.home() / ".secureops_vault.json"
KDF_ITERATIONS = 600_000
KEYWORDS = ("error", "warning", "failed", "success", "critical")


# ============================================================
# PANEL 1: FILE ANALYZER (TXT, LOG, CSV)
# ============================================================

class FileAnalyzer:
    """Scan text, log, and CSV files in chunks/lines."""

    def analyze(self, paths, progress_callback, cancel_event):
        word_counts = Counter()
        unique_words = set()
        keyword_counts = Counter()
        total_lines = 0
        total_bytes = 0
        files_scanned = 0
        errors = []

        for path in paths:
            if cancel_event.is_set():
                break

            try:
                file_size = path.stat().st_size
                bytes_read = 0

                with path.open("r", encoding="utf-8-sig", errors="replace", newline="") as file:
                    for line in file:
                        if cancel_event.is_set():
                            break

                        total_lines += 1
                        line_bytes = len(line.encode("utf-8"))
                        bytes_read += line_bytes
                        total_bytes += line_bytes

                        words = re.findall(r"\b[\w'-]+\b", line.lower())
                        word_counts.update(words)
                        unique_words.update(words)

                        lowered = line.lower()
                        for keyword in KEYWORDS:
                            keyword_counts[keyword] += lowered.count(keyword)

                        progress_callback(path.name, bytes_read, file_size, total_bytes)

                if not cancel_event.is_set():
                    files_scanned += 1

            except (OSError, UnicodeError, PermissionError) as exc:
                errors.append(f"{path}: {exc}")

        return {
            "files_scanned": files_scanned,
            "lines": total_lines,
            "bytes": total_bytes,
            "unique_words": len(unique_words),
            "top_words": word_counts.most_common(15),
            "keywords": dict(keyword_counts),
            "errors": errors,
            "cancelled": cancel_event.is_set(),
        }


# ============================================================
# PANEL 2: CSV MANAGER
# ============================================================

class CSVManager:
    """Create, load, edit, and save CSV data."""

    def __init__(self):
        self.file_path = None
        self.headers = []
        self.rows = []

    def create_new(self, headers):
        cleaned = [h.strip() for h in headers if h.strip()]
        if not cleaned:
            raise ValueError("Enter at least one column name.")
        if len(set(cleaned)) != len(cleaned):
            raise ValueError("Column names must be unique.")
        self.file_path = None
        self.headers = cleaned
        self.rows = []

    def open_csv(self, file_path):
        path = Path(file_path)
        if path.suffix.lower() != ".csv":
            raise ValueError("Please select a .csv file.")

        with path.open("r", newline="", encoding="utf-8-sig") as file:
            reader = csv.reader(file)
            all_rows = list(reader)

        if not all_rows:
            raise ValueError("The selected CSV file is empty.")

        headers = all_rows[0]
        if not headers or any(not h.strip() for h in headers):
            raise ValueError("CSV header row contains an empty column name.")
        if len(set(headers)) != len(headers):
            raise ValueError("CSV contains duplicate column names.")

        self.headers = headers
        self.rows = []
        for row in all_rows[1:]:
            # Keep rows rectangular for Treeview and future CSV writes.
            row = row[:len(headers)] + [""] * max(0, len(headers) - len(row))
            self.rows.append(row)
        self.file_path = path

    def add_row(self, values):
        if not self.headers:
            raise ValueError("Create or open a CSV first.")
        if len(values) != len(self.headers):
            raise ValueError("Enter a value for each column.")
        self.rows.append([str(v) for v in values])

    def update_cell(self, row_index, column_index, value):
        self.rows[row_index][column_index] = value

    def save(self, file_path=None):
        destination = Path(file_path) if file_path else self.file_path
        if destination is None:
            raise ValueError("Choose a filename using Save As.")

        if destination.suffix.lower() != ".csv":
            destination = destination.with_suffix(".csv")

        # Write to a temporary file, then replace destination.
        temporary = destination.with_suffix(destination.suffix + ".tmp")
        try:
            with temporary.open("w", newline="", encoding="utf-8-sig") as file:
                writer = csv.writer(file)
                writer.writerow(self.headers)
                writer.writerows(self.rows)
                file.flush()
                os.fsync(file.fileno())
            os.replace(temporary, destination)
        finally:
            if temporary.exists():
                temporary.unlink()

        self.file_path = destination
        return destination


# ============================================================
# PANEL 3: ENCRYPTED VAULT
# ============================================================

class UserVault:
    """Encrypt all vault records with a password-derived Fernet key."""

    def __init__(self, file_path=VAULT_FILE):
        self.__file_path = Path(file_path)
        self.__fernet = None
        self.__records = {}

    @property
    def exists(self):
        return self.__file_path.exists()

    def __derive_key(self, password, salt):
        kdf = PBKDF2HMAC(
            algorithm=hashes.SHA256(),
            length=32,
            salt=salt,
            iterations=KDF_ITERATIONS,
        )
        return base64.urlsafe_b64encode(kdf.derive(password.encode("utf-8")))

    def create(self, password):
        if len(password) < 8:
            raise ValueError("Master password must contain at least 8 characters.")
        if self.exists:
            raise FileExistsError("A vault already exists.")
        salt = secrets.token_bytes(16)
        self.__fernet = Fernet(self.__derive_key(password, salt))
        self.__records = {}
        self.__save(salt)

    def unlock(self, password):
        if not self.exists:
            raise FileNotFoundError("No vault exists yet.")
        if self.__file_path.stat().st_size > 20 * 1024 * 1024:
            raise ValueError("Vault file is too large.")

        with self.__file_path.open("r", encoding="utf-8") as file:
            envelope = json.load(file)

        salt = base64.b64decode(envelope["salt"], validate=True)
        if len(salt) != 16:
            raise ValueError("Invalid vault salt.")

        candidate = Fernet(self.__derive_key(password, salt))
        try:
            plaintext = candidate.decrypt(envelope["data"].encode("ascii"))
            records = json.loads(plaintext.decode("utf-8"))
        except (InvalidToken, UnicodeError, json.JSONDecodeError):
            raise ValueError("Incorrect password or corrupted vault.") from None

        if not isinstance(records, dict):
            raise ValueError("Invalid vault data.")
        self.__fernet = candidate
        self.__records = records

    def __save(self, salt):
        if self.__fernet is None:
            raise PermissionError("Unlock the vault first.")
        plaintext = json.dumps(self.__records, ensure_ascii=False).encode("utf-8")
        envelope = {
            "salt": base64.b64encode(salt).decode("ascii"),
            "data": self.__fernet.encrypt(plaintext).decode("ascii"),
        }
        temporary = self.__file_path.with_suffix(".tmp")
        try:
            with temporary.open("w", encoding="utf-8") as file:
                json.dump(envelope, file)
                file.flush()
                os.fsync(file.fileno())
            os.replace(temporary, self.__file_path)
        finally:
            if temporary.exists():
                temporary.unlink()

    def __get_salt(self):
        with self.__file_path.open("r", encoding="utf-8") as file:
            return base64.b64decode(json.load(file)["salt"], validate=True)

    def add_record(self, title, username, secret_text, notes=""):
        if self.__fernet is None:
            raise PermissionError("Unlock the vault first.")
        title = title.strip()
        if not title:
            raise ValueError("Record title cannot be empty.")
        if title in self.__records:
            raise ValueError("A record with this title already exists.")
        self.__records[title] = {
            "username": username,
            "secret": secret_text,
            "notes": notes,
        }
        self.__save(self.__get_salt())

    def get_record(self, title):
        if self.__fernet is None:
            raise PermissionError("Unlock the vault first.")
        return self.__records[title].copy()

    def list_titles(self):
        if self.__fernet is None:
            raise PermissionError("Unlock the vault first.")
        return sorted(self.__records, key=str.lower)

    def delete_record(self, title):
        if self.__fernet is None:
            raise PermissionError("Unlock the vault first.")
        del self.__records[title]
        self.__save(self.__get_salt())

    def lock(self):
        self.__fernet = None
        self.__records.clear()


# ============================================================
# MAIN TKINTER APPLICATION
# ============================================================

class SecureOpsApp:
    def __init__(self, root):
        self.root = root
        self.root.title("SecureOps | Local File Intelligence")
        self.root.geometry("1150x780")
        self.root.minsize(900, 620)

        self.analyzer = FileAnalyzer()
        self.csv_manager = CSVManager()
        self.vault = UserVault()

        self.file_queue = queue.Queue()
        self.cancel_event = threading.Event()
        self.scan_thread = None
        self.selected_paths = []

        self.stack = []
        self.queue_items = []

        self.setup_style()
        self.build_interface()
        self.root.protocol("WM_DELETE_WINDOW", self.close_app)
        self.root.after(100, self.process_queue)

    def setup_style(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass
        style.configure("Title.TLabel", font=("Arial", 18, "bold"))
        style.configure("Heading.TLabel", font=("Arial", 12, "bold"))
        style.configure("TButton", padding=5)
        style.configure("Treeview", rowheight=24)

    def build_interface(self):
        header = ttk.Frame(self.root, padding=12)
        header.pack(fill=tk.X)
        ttk.Label(header, text="SECUREOPS", style="Title.TLabel").pack(side=tk.LEFT)
        ttk.Label(
            header, text="File Intelligence | CSV Manager | Encrypted Vault | Data Structures"
        ).pack(side=tk.RIGHT)

        self.tabs = ttk.Notebook(self.root)
        self.tabs.pack(fill=tk.BOTH, expand=True, padx=12, pady=(0, 8))

        self.scanner_tab = ttk.Frame(self.tabs, padding=10)
        self.csv_tab = ttk.Frame(self.tabs, padding=10)
        self.vault_tab = ttk.Frame(self.tabs, padding=10)
        self.structures_tab = ttk.Frame(self.tabs, padding=10)

        self.tabs.add(self.scanner_tab, text="File Intelligence")
        self.tabs.add(self.csv_tab, text="CSV Manager")
        self.tabs.add(self.vault_tab, text="Secure Vault")
        self.tabs.add(self.structures_tab, text="Stack & Queue Lab")

        self.build_scanner_tab()
        self.build_csv_tab()
        self.build_vault_tab()
        self.build_structures_tab()

        self.status = tk.StringVar(value="Ready")
        ttk.Label(self.root, textvariable=self.status, anchor="w", padding=6).pack(fill=tk.X)

    # ---------------- File Intelligence ----------------

    def build_scanner_tab(self):
        ttk.Label(
            self.scanner_tab, text="Multi-Threaded Text, Log & CSV Analyzer",
            style="Heading.TLabel"
        ).pack(anchor="w", pady=(0, 8))

        bar = ttk.Frame(self.scanner_tab)
        bar.pack(fill=tk.X, pady=4)

        ttk.Button(bar, text="Select File", command=self.select_scan_file).pack(side=tk.LEFT, padx=3)
        ttk.Button(bar, text="Select Folder", command=self.select_scan_folder).pack(side=tk.LEFT, padx=3)
        self.scan_button = ttk.Button(bar, text="Start Scan", command=self.start_scan)
        self.scan_button.pack(side=tk.LEFT, padx=3)
        self.cancel_button = ttk.Button(bar, text="Cancel", command=self.cancel_scan, state="disabled")
        self.cancel_button.pack(side=tk.LEFT, padx=3)

        self.scan_path_label = ttk.Label(self.scanner_tab, text="No files selected", wraplength=1000)
        self.scan_path_label.pack(anchor="w", pady=6)

        self.progress = ttk.Progressbar(self.scanner_tab, mode="indeterminate")
        self.progress.pack(fill=tk.X, pady=4)

        self.scan_output = tk.Text(self.scanner_tab, wrap="word", font=("Consolas", 10))
        self.scan_output.pack(fill=tk.BOTH, expand=True, pady=6)
        self.scan_output.insert(tk.END, "Select .txt, .log, or .csv files to analyze.\n")
        self.scan_output.config(state="disabled")

    def select_scan_file(self):
        filename = filedialog.askopenfilename(
            title="Select file",
            filetypes=[("Supported files", "*.txt *.log *.csv"), ("All files", "*.*")]
        )
        if filename:
            path = Path(filename)
            if path.suffix.lower() not in (".txt", ".log", ".csv"):
                messagebox.showwarning("Unsupported", "Choose a .txt, .log, or .csv file.")
                return
            self.selected_paths = [path]
            self.scan_path_label.config(text=str(path))

    def select_scan_folder(self):
        folder = filedialog.askdirectory(title="Select folder")
        if folder:
            try:
                self.selected_paths = sorted(
                    p for p in Path(folder).rglob("*")
                    if p.is_file() and p.suffix.lower() in (".txt", ".log", ".csv")
                )
                self.scan_path_label.config(
                    text=f"{folder}\n{len(self.selected_paths)} supported file(s) found"
                )
            except OSError as exc:
                messagebox.showerror("Folder error", str(exc))

    def start_scan(self):
        if self.scan_thread and self.scan_thread.is_alive():
            messagebox.showinfo("Scan running", "A scan is already running.")
            return
        if not self.selected_paths:
            messagebox.showwarning("No files", "Select a file or folder first.")
            return

        self.cancel_event.clear()
        self.scan_button.config(state="disabled")
        self.cancel_button.config(state="normal")
        self.progress.start(10)
        self.status.set("Scanning in background...")
        self.scan_output.config(state="normal")
        self.scan_output.delete("1.0", tk.END)
        self.scan_output.insert(tk.END, "Scan started...\n")
        self.scan_output.config(state="disabled")

        self.scan_thread = threading.Thread(
            target=self.scan_worker, args=(list(self.selected_paths),), daemon=True
        )
        self.scan_thread.start()

    def scan_worker(self, paths):
        try:
            def progress_callback(name, current, size, total):
                self.file_queue.put(("progress", name, current, size, total))
            result = self.analyzer.analyze(paths, progress_callback, self.cancel_event)
            self.file_queue.put(("result", result))
        except Exception as exc:
            self.file_queue.put(("error", str(exc)))

    def cancel_scan(self):
        self.cancel_event.set()
        self.status.set("Cancellation requested...")

    def process_queue(self):
        try:
            while True:
                item = self.file_queue.get_nowait()
                if item[0] == "progress":
                    _, name, current, size, total = item
                    self.status.set(f"Scanning {name}: {current:,}/{size:,} bytes")
                elif item[0] == "result":
                    self.progress.stop()
                    self.scan_button.config(state="normal")
                    self.cancel_button.config(state="disabled")
                    self.display_scan_result(item[1])
                elif item[0] == "error":
                    self.progress.stop()
                    self.scan_button.config(state="normal")
                    self.cancel_button.config(state="disabled")
                    messagebox.showerror("Scan error", item[1])
        except queue.Empty:
            pass
        self.root.after(100, self.process_queue)

    def display_scan_result(self, result):
        self.scan_output.config(state="normal")
        self.scan_output.delete("1.0", tk.END)
        self.scan_output.insert(tk.END, "========== FILE ANALYSIS REPORT ==========\n\n")
        self.scan_output.insert(tk.END, f"Files scanned: {result['files_scanned']}\n")
        self.scan_output.insert(tk.END, f"Lines: {result['lines']:,}\n")
        self.scan_output.insert(tk.END, f"Bytes processed: {result['bytes']:,}\n")
        self.scan_output.insert(tk.END, f"Unique words: {result['unique_words']:,}\n")
        self.scan_output.insert(tk.END, "\n--- KEYWORD COUNTS ---\n")
        for word, count in result["keywords"].items():
            self.scan_output.insert(tk.END, f"{word}: {count}\n")
        self.scan_output.insert(tk.END, "\n--- TOP 15 WORDS ---\n")
        for word, count in result["top_words"]:
            self.scan_output.insert(tk.END, f"{word:<22} {count}\n")
        if result["errors"]:
            self.scan_output.insert(tk.END, "\n--- ERRORS ---\n")
            for error in result["errors"]:
                self.scan_output.insert(tk.END, error + "\n")
        if result["cancelled"]:
            self.scan_output.insert(tk.END, "\nScan cancelled.\n")
            self.status.set("Scan cancelled")
        else:
            self.status.set("File analysis completed")
        self.scan_output.config(state="disabled")

    # ---------------- CSV Manager ----------------

    def build_csv_tab(self):
        ttk.Label(
            self.csv_tab, text="Create, Open, View and Edit CSV Files",
            style="Heading.TLabel"
        ).pack(anchor="w", pady=(0, 8))

        toolbar = ttk.Frame(self.csv_tab)
        toolbar.pack(fill=tk.X, pady=4)

        ttk.Button(toolbar, text="New CSV", command=self.new_csv_dialog).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Open CSV", command=self.open_csv_dialog).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Add Row", command=self.add_csv_row_dialog).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Edit Selected Cell", command=self.edit_csv_cell).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Delete Selected Row", command=self.delete_csv_row).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Save", command=self.save_csv).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Save As", command=self.save_csv_as).pack(side=tk.LEFT, padx=3)
        ttk.Button(toolbar, text="Create Sample CSV", command=self.create_sample_csv).pack(side=tk.LEFT, padx=3)

        self.csv_path_label = ttk.Label(self.csv_tab, text="No CSV open", wraplength=1000)
        self.csv_path_label.pack(anchor="w", pady=6)

        table_frame = ttk.Frame(self.csv_tab)
        table_frame.pack(fill=tk.BOTH, expand=True)

        self.csv_tree = ttk.Treeview(table_frame, show="headings", selectmode="browse")
        yscroll = ttk.Scrollbar(table_frame, orient="vertical", command=self.csv_tree.yview)
        xscroll = ttk.Scrollbar(table_frame, orient="horizontal", command=self.csv_tree.xview)
        self.csv_tree.configure(yscrollcommand=yscroll.set, xscrollcommand=xscroll.set)
        self.csv_tree.grid(row=0, column=0, sticky="nsew")
        yscroll.grid(row=0, column=1, sticky="ns")
        xscroll.grid(row=1, column=0, sticky="ew")
        table_frame.rowconfigure(0, weight=1)
        table_frame.columnconfigure(0, weight=1)

        self.csv_info = tk.StringVar(value="0 rows")
        ttk.Label(self.csv_tab, textvariable=self.csv_info).pack(anchor="w", pady=5)

    def refresh_csv_table(self):
        self.csv_tree.delete(*self.csv_tree.get_children())
        self.csv_tree["columns"] = self.csv_manager.headers

        for header in self.csv_manager.headers:
            self.csv_tree.heading(header, text=header)
            self.csv_tree.column(header, width=150, minwidth=80, stretch=True)

        for index, row in enumerate(self.csv_manager.rows):
            self.csv_tree.insert("", tk.END, iid=str(index), values=row)

        self.csv_info.set(f"{len(self.csv_manager.rows)} data row(s), {len(self.csv_manager.headers)} column(s)")
        self.csv_path_label.config(
            text=f"Current file: {self.csv_manager.file_path}"
            if self.csv_manager.file_path else "New CSV (not saved yet)"
        )

    def new_csv_dialog(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Create New CSV")
        dialog.transient(self.root)
        dialog.grab_set()
        ttk.Label(dialog, text="Enter column names separated by commas:").pack(
            padx=12, pady=(12, 4), anchor="w"
        )
        entry_var = tk.StringVar(value="Name,Department,Status")
        entry = ttk.Entry(dialog, textvariable=entry_var, width=55)
        entry.pack(padx=12, pady=4)
        entry.focus_set()

        def create():
            try:
                self.csv_manager.create_new(entry_var.get().split(","))
                self.refresh_csv_table()
                dialog.destroy()
                self.status.set("New CSV created; use Save As to save it.")
            except Exception as exc:
                messagebox.showerror("CSV error", str(exc), parent=dialog)

        ttk.Button(dialog, text="Create", command=create).pack(pady=12)
        dialog.bind("<Return>", lambda event: create())

    def open_csv_dialog(self):
        filename = filedialog.askopenfilename(
            title="Open CSV",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )
        if not filename:
            return
        try:
            self.csv_manager.open_csv(filename)
            self.refresh_csv_table()
            self.status.set(f"Opened {Path(filename).name}")
        except Exception as exc:
            messagebox.showerror("Open CSV error", str(exc))

    def add_csv_row_dialog(self):
        if not self.csv_manager.headers:
            messagebox.showwarning("No CSV", "Create or open a CSV first.")
            return

        dialog = tk.Toplevel(self.root)
        dialog.title("Add CSV Row")
        dialog.transient(self.root)
        dialog.grab_set()
        variables = []

        form = ttk.Frame(dialog, padding=12)
        form.pack(fill=tk.BOTH, expand=True)

        for row_index, header in enumerate(self.csv_manager.headers):
            ttk.Label(form, text=header).grid(row=row_index, column=0, sticky="w", padx=5, pady=4)
            variable = tk.StringVar()
            ttk.Entry(form, textvariable=variable, width=35).grid(
                row=row_index, column=1, sticky="ew", padx=5, pady=4
            )
            variables.append(variable)

        def add():
            try:
                self.csv_manager.add_row([v.get() for v in variables])
                self.refresh_csv_table()
                dialog.destroy()
                self.status.set("Row added. Save to keep changes.")
            except Exception as exc:
                messagebox.showerror("Add row error", str(exc), parent=dialog)

        ttk.Button(form, text="Add Row", command=add).grid(
            row=len(variables), column=0, columnspan=2, pady=10
        )
        dialog.bind("<Return>", lambda event: add())

    def edit_csv_cell(self):
        selected = self.csv_tree.selection()
        if not selected:
            messagebox.showwarning("Select row", "Select a row first.")
            return
        if not self.csv_manager.headers:
            return

        row_index = int(selected[0])
        dialog = tk.Toplevel(self.root)
        dialog.title("Edit Selected Row")
        dialog.transient(self.root)
        dialog.grab_set()
        variables = []

        form = ttk.Frame(dialog, padding=12)
        form.pack(fill=tk.BOTH, expand=True)

        for col_index, header in enumerate(self.csv_manager.headers):
            ttk.Label(form, text=header).grid(row=col_index, column=0, sticky="w", padx=5, pady=4)
            variable = tk.StringVar(value=self.csv_manager.rows[row_index][col_index])
            ttk.Entry(form, textvariable=variable, width=35).grid(
                row=col_index, column=1, sticky="ew", padx=5, pady=4
            )
            variables.append(variable)

        def save_edit():
            self.csv_manager.rows[row_index] = [v.get() for v in variables]
            self.refresh_csv_table()
            dialog.destroy()
            self.status.set("Row updated. Save to keep changes.")

        ttk.Button(form, text="Apply Changes", command=save_edit).grid(
            row=len(variables), column=0, columnspan=2, pady=10
        )

    def delete_csv_row(self):
        selected = self.csv_tree.selection()
        if not selected:
            messagebox.showwarning("Select row", "Select a row to delete.")
            return
        row_index = int(selected[0])
        if messagebox.askyesno("Delete row", "Delete the selected row?"):
            del self.csv_manager.rows[row_index]
            self.refresh_csv_table()
            self.status.set("Row deleted. Save to keep changes.")

    def save_csv(self):
        try:
            if not self.csv_manager.headers:
                raise ValueError("Create or open a CSV first.")
            if self.csv_manager.file_path is None:
                self.save_csv_as()
                return
            destination = self.csv_manager.save()
            self.refresh_csv_table()
            self.status.set(f"Saved {destination}")
            messagebox.showinfo("Saved", f"CSV saved to:\n{destination}")
        except Exception as exc:
            messagebox.showerror("Save CSV error", str(exc))

    def save_csv_as(self):
        if not self.csv_manager.headers:
            messagebox.showwarning("No CSV", "Create or open a CSV first.")
            return
        filename = filedialog.asksaveasfilename(
            title="Save CSV As",
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv")],
            initialfile="secureops_data.csv"
        )
        if not filename:
            return
        try:
            destination = self.csv_manager.save(filename)
            self.refresh_csv_table()
            self.status.set(f"Saved {destination}")
            messagebox.showinfo("Saved", f"CSV saved to:\n{destination}")
        except Exception as exc:
            messagebox.showerror("Save CSV error", str(exc))

    def create_sample_csv(self):
        try:
            self.csv_manager.create_new(["Name", "Department", "Status"])
            self.csv_manager.add_row(["Alice", "IT", "Success"])
            self.csv_manager.add_row(["Bob", "EEE", "Warning"])
            self.csv_manager.add_row(["Charlie", "Data Science", "Error"])
            self.refresh_csv_table()
            self.status.set("Sample data created. Click Save As to save as CSV.")
        except Exception as exc:
            messagebox.showerror("Sample CSV error", str(exc))

    # ---------------- Secure Vault ----------------

    def build_vault_tab(self):
        ttk.Label(
            self.vault_tab, text="Encrypted Local Credential & Notes Vault",
            style="Heading.TLabel"
        ).pack(anchor="w", pady=(0, 8))

        self.vault_state = tk.StringVar()
        self.refresh_vault_state()
        ttk.Label(self.vault_tab, textvariable=self.vault_state, wraplength=1000).pack(anchor="w", pady=4)

        auth = ttk.LabelFrame(self.vault_tab, text="Master Password", padding=8)
        auth.pack(fill=tk.X, pady=8)
        self.master_password = tk.StringVar()
        ttk.Entry(auth, textvariable=self.master_password, show="*", width=30).pack(side=tk.LEFT, padx=4)
        ttk.Button(auth, text="Create Vault", command=self.create_vault).pack(side=tk.LEFT, padx=4)
        ttk.Button(auth, text="Unlock Vault", command=self.unlock_vault).pack(side=tk.LEFT, padx=4)
        ttk.Button(auth, text="Lock Vault", command=self.lock_vault).pack(side=tk.LEFT, padx=4)

        self.vault_controls = ttk.Frame(self.vault_tab)
        self.vault_controls.pack(fill=tk.BOTH, expand=True, pady=6)

        form = ttk.LabelFrame(self.vault_controls, text="Add Record", padding=8)
        form.pack(side=tk.LEFT, fill=tk.Y, padx=(0, 8))
        self.record_title = tk.StringVar()
        self.record_username = tk.StringVar()
        self.record_secret = tk.StringVar()

        for label, variable, hide in [
            ("Record title", self.record_title, False),
            ("Username / email", self.record_username, False),
            ("Password / secret", self.record_secret, True),
        ]:
            ttk.Label(form, text=label).pack(anchor="w", pady=(4, 2))
            ttk.Entry(form, textvariable=variable, show="*" if hide else "", width=28).pack(fill=tk.X)

        ttk.Label(form, text="Notes").pack(anchor="w", pady=(4, 2))
        self.record_notes = tk.Text(form, width=28, height=5)
        self.record_notes.pack(fill=tk.X)
        ttk.Button(form, text="Save Encrypted Record", command=self.add_vault_record).pack(fill=tk.X, pady=8)

        records_frame = ttk.LabelFrame(self.vault_controls, text="Vault Records", padding=8)
        records_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        self.record_list = tk.Listbox(records_frame)
        self.record_list.pack(fill=tk.BOTH, expand=True)
        self.record_list.bind("<Double-Button-1>", lambda event: self.view_vault_record())
        buttons = ttk.Frame(records_frame)
        buttons.pack(fill=tk.X, pady=5)
        ttk.Button(buttons, text="View", command=self.view_vault_record).pack(side=tk.LEFT, padx=3)
        ttk.Button(buttons, text="Delete", command=self.delete_vault_record).pack(side=tk.LEFT, padx=3)

        self.set_vault_controls(False)

    def refresh_vault_state(self):
        self.vault_state.set(
            f"Vault file: {VAULT_FILE}\n"
            + ("Enter master password to unlock." if self.vault.exists
               else "No vault found. Create one with a strong master password.")
        )

    def ask_password_confirmation(self):
        dialog = tk.Toplevel(self.root)
        dialog.title("Confirm Master Password")
        dialog.transient(self.root)
        dialog.grab_set()
        result = {"value": None}
        value = tk.StringVar()
        ttk.Label(dialog, text="Re-enter master password:").pack(padx=14, pady=(12, 4))
        entry = ttk.Entry(dialog, textvariable=value, show="*", width=30)
        entry.pack(padx=14, pady=4)
        entry.focus_set()

        def submit():
            result["value"] = value.get()
            dialog.destroy()

        ttk.Button(dialog, text="Confirm", command=submit).pack(pady=10)
        dialog.bind("<Return>", lambda event: submit())
        self.root.wait_window(dialog)
        return result["value"]

    def create_vault(self):
        password = self.master_password.get()
        confirm = self.ask_password_confirmation()
        if confirm is None:
            return
        if password != confirm:
            messagebox.showerror("Mismatch", "Passwords do not match.")
            return
        try:
            self.vault.create(password)
            self.master_password.set("")
            self.set_vault_controls(True)
            self.refresh_record_list()
            self.refresh_vault_state()
            messagebox.showinfo("Vault", "Encrypted vault created.")
        except Exception as exc:
            messagebox.showerror("Vault error", str(exc))

    def unlock_vault(self):
        try:
            self.vault.unlock(self.master_password.get())
            self.master_password.set("")
            self.set_vault_controls(True)
            self.refresh_record_list()
            self.refresh_vault_state()
            messagebox.showinfo("Vault", "Vault unlocked.")
        except Exception as exc:
            messagebox.showerror("Unlock error", str(exc))

    def lock_vault(self):
        self.vault.lock()
        self.set_vault_controls(False)
        self.record_list.delete(0, tk.END)
        self.master_password.set("")
        self.refresh_vault_state()

    def set_vault_controls(self, enabled):
        state = "normal" if enabled else "disabled"
        for widget in self.vault_controls.winfo_children():
            self.set_widget_state(widget, state)

    def set_widget_state(self, widget, state):
        try:
            widget.configure(state=state)
        except tk.TclError:
            pass
        for child in widget.winfo_children():
            self.set_widget_state(child, state)

    def refresh_record_list(self):
        self.record_list.delete(0, tk.END)
        for title in self.vault.list_titles():
            self.record_list.insert(tk.END, title)

    def add_vault_record(self):
        try:
            self.vault.add_record(
                self.record_title.get(), self.record_username.get(),
                self.record_secret.get(), self.record_notes.get("1.0", tk.END).strip()
            )
            self.record_title.set("")
            self.record_username.set("")
            self.record_secret.set("")
            self.record_notes.delete("1.0", tk.END)
            self.refresh_record_list()
            messagebox.showinfo("Saved", "Encrypted record saved.")
        except Exception as exc:
            messagebox.showerror("Save error", str(exc))

    def selected_record_title(self):
        selection = self.record_list.curselection()
        if not selection:
            messagebox.showwarning("Select record", "Choose a record first.")
            return None
        return self.record_list.get(selection[0])

    def view_vault_record(self):
        title = self.selected_record_title()
        if title is None:
            return
        try:
            record = self.vault.get_record(title)
            dialog = tk.Toplevel(self.root)
            dialog.title("Vault Record")
            dialog.geometry("430x330")
            ttk.Label(dialog, text=title, style="Heading.TLabel").pack(anchor="w", padx=12, pady=8)
            ttk.Label(dialog, text=f"Username: {record['username']}").pack(anchor="w", padx=12, pady=4)
            ttk.Label(dialog, text="Secret:").pack(anchor="w", padx=12, pady=(6, 2))
            secret_entry = ttk.Entry(dialog, width=45, show="*")
            secret_entry.pack(fill=tk.X, padx=12)
            secret_entry.insert(0, record["secret"])
            show = tk.BooleanVar(value=False)
            ttk.Checkbutton(
                dialog, text="Show secret", variable=show,
                command=lambda: secret_entry.configure(show="" if show.get() else "*")
            ).pack(anchor="w", padx=12, pady=4)
            ttk.Label(dialog, text="Notes:").pack(anchor="w", padx=12, pady=(6, 2))
            notes = tk.Text(dialog, height=6, wrap="word")
            notes.pack(fill=tk.BOTH, expand=True, padx=12, pady=(0, 12))
            notes.insert("1.0", record["notes"])
            notes.config(state="disabled")
        except Exception as exc:
            messagebox.showerror("Read error", str(exc))

    def delete_vault_record(self):
        title = self.selected_record_title()
        if title is None:
            return
        if messagebox.askyesno("Delete", f"Delete '{title}' permanently?"):
            try:
                self.vault.delete_record(title)
                self.refresh_record_list()
            except Exception as exc:
                messagebox.showerror("Delete error", str(exc))

    # ---------------- Stack and Queue Lab ----------------

    def build_structures_tab(self):
        ttk.Label(
            self.structures_tab, text="Interactive Stack & Queue Visual Lab",
            style="Heading.TLabel"
        ).pack(anchor="w", pady=(0, 8))

        main = ttk.Frame(self.structures_tab)
        main.pack(fill=tk.BOTH, expand=True)

        stack_frame = ttk.LabelFrame(main, text="Stack — LIFO", padding=8)
        stack_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 6))
        ttk.Label(stack_frame, text="Last In, First Out").pack()
        self.stack_input = tk.StringVar()
        ttk.Entry(stack_frame, textvariable=self.stack_input).pack(fill=tk.X, pady=5)
        buttons = ttk.Frame(stack_frame)
        buttons.pack(fill=tk.X)
        ttk.Button(buttons, text="Push", command=self.push_stack).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        ttk.Button(buttons, text="Pop", command=self.pop_stack).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        self.stack_canvas = tk.Canvas(stack_frame, height=380, background="white", highlightthickness=1)
        self.stack_canvas.pack(fill=tk.BOTH, expand=True, pady=8)

        queue_frame = ttk.LabelFrame(main, text="Queue — FIFO", padding=8)
        queue_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(6, 0))
        ttk.Label(queue_frame, text="First In, First Out").pack()
        self.queue_input = tk.StringVar()
        ttk.Entry(queue_frame, textvariable=self.queue_input).pack(fill=tk.X, pady=5)
        buttons = ttk.Frame(queue_frame)
        buttons.pack(fill=tk.X)
        ttk.Button(buttons, text="Enqueue", command=self.enqueue).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        ttk.Button(buttons, text="Dequeue", command=self.dequeue).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        self.queue_canvas = tk.Canvas(queue_frame, height=380, background="white", highlightthickness=1)
        self.queue_canvas.pack(fill=tk.BOTH, expand=True, pady=8)

        self.stack_canvas.bind("<Configure>", lambda event: self.draw_stack())
        self.queue_canvas.bind("<Configure>", lambda event: self.draw_queue())
        self.draw_stack()
        self.draw_queue()

    def push_stack(self):
        value = self.stack_input.get().strip()
        if not value:
            messagebox.showwarning("Missing item", "Enter a stack item.")
            return
        if len(self.stack) >= 12:
            messagebox.showwarning("Stack full", "Maximum 12 items.")
            return
        self.stack.append(value)
        self.stack_input.set("")
        self.draw_stack()

    def pop_stack(self):
        if not self.stack:
            messagebox.showinfo("Empty stack", "Nothing to pop.")
            return
        value = self.stack.pop()
        self.draw_stack()
        self.status.set(f"Popped from stack: {value}")

    def enqueue(self):
        value = self.queue_input.get().strip()
        if not value:
            messagebox.showwarning("Missing item", "Enter a queue item.")
            return
        if len(self.queue_items) >= 12:
            messagebox.showwarning("Queue full", "Maximum 12 items.")
            return
        self.queue_items.append(value)
        self.queue_input.set("")
        self.draw_queue()

    def dequeue(self):
        if not self.queue_items:
            messagebox.showinfo("Empty queue", "Nothing to dequeue.")
            return
        value = self.queue_items.pop(0)
        self.draw_queue()
        self.status.set(f"Dequeued from queue: {value}")

    def draw_stack(self):
        if not hasattr(self, "stack_canvas"):
            return
        canvas = self.stack_canvas
        canvas.delete("all")
        width = max(canvas.winfo_width(), 250)
        height = max(canvas.winfo_height(), 300)
        canvas.create_text(width // 2, 16, text="TOP", font=("Arial", 10, "bold"))
        box_width, box_height = min(width - 25, 230), 25
        start_y = height - 34
        canvas.create_line(width // 2 - box_width // 2 - 5, height - 10,
                           width // 2 + box_width // 2 + 5, height - 10, width=3)
        for index, value in enumerate(reversed(self.stack)):
            y = start_y - index * (box_height + 5)
            if y < 34:
                break
            canvas.create_rectangle(width // 2 - box_width // 2, y - box_height,
                                    width // 2 + box_width // 2, y,
                                    outline="#25476a", fill="#cfe8ff", width=2)
            canvas.create_text(width // 2, y - box_height // 2, text=value[:28])
        canvas.create_text(width // 2, height - 22, text=f"Stack size: {len(self.stack)}")

    def draw_queue(self):
        if not hasattr(self, "queue_canvas"):
            return
        canvas = self.queue_canvas
        canvas.delete("all")
        width = max(canvas.winfo_width(), 250)
        height = max(canvas.winfo_height(), 300)
        canvas.create_text(width // 2, 18, text="FRONT  →  REAR", font=("Arial", 10, "bold"))
        visible = self.queue_items[:12]
        box_width, box_height, gap = min(90, max(55, (width - 35) // 2)), 40, 6
        y = height // 2
        total_width = len(visible) * (box_width + gap) - gap
        start_x = max(10, (width - total_width) // 2)
        for index, value in enumerate(visible):
            x = start_x + index * (box_width + gap)
            canvas.create_rectangle(x, y - box_height // 2, x + box_width, y + box_height // 2,
                                    outline="#276749", fill="#d1fae5", width=2)
            canvas.create_text(x + box_width // 2, y, text=value[:12])
        if not visible:
            canvas.create_text(width // 2, y, text="Queue is empty")
        canvas.create_text(width // 2, height - 25, text=f"Queue size: {len(self.queue_items)}")

    # ---------------- Close application ----------------

    def close_app(self):
        if self.scan_thread and self.scan_thread.is_alive():
            self.cancel_event.set()
        self.vault.lock()
        self.root.destroy()


if __name__ == "__main__":
    root = tk.Tk()
    app = SecureOpsApp(root)
    root.mainloop()
'''
Path('secureops.py').write_text(source_code, encoding='utf-8')
print('Created secureops.py with', len(source_code.splitlines()), 'lines')

## Launch the application

Run this cell after the source-setup cell. The app will open in a separate window.

In [ ]:
%run secureops.py